# 06 · Track record review

Load the published site files, verify the hash chain, see that editing one old forecast breaks it, and read scores against the market. Run this weekly once the pipeline is live — or right now, against the fictional demo data from `make demo`.

In [ ]:
import sys

sys.path.insert(0, "../src")
import json
import os

from forecaster import trackrecord

In [ ]:
public_dir = "../public"
if os.path.exists(f"{public_dir}/track-record.json"):
    track_record = json.load(open(f"{public_dir}/track-record.json"))
    manifest = json.load(open(f"{public_dir}/manifest.json"))
    print(f'{len(track_record)} questions published, generated at {manifest["generated_at"]}')
else:
    print("run `make demo` first to generate public/track-record.json")
    track_record, manifest = [], {}

## Verifying the manifest's SHA-256 checksums

The same check the `pages` workflow runs before publishing: every file's hash must match the manifest, or publishing stops.

In [ ]:
from pathlib import Path

if manifest:
    for name, expected in manifest["files"].items():
        actual = trackrecord.sha256_file(Path(public_dir) / name)
        status = "OK" if actual == expected else "MISMATCH"
        print(f"{name}: {status}")

## Verifying the forecast hash chain

Each forecast's hash covers its own fields plus the previous forecast's hash. Editing any past forecast without recomputing every hash after it breaks the chain — `verify_chain` returns the exact index where it broke.

In [ ]:
from dataclasses import dataclass
from datetime import datetime


@dataclass
class _Row:
    question_id: str
    created_at: datetime
    model: str
    prompt_version: str
    prompt_sha256: str
    raw_prob: float
    calibrated_prob: float
    calibrator_version: str
    prev_hash: str = ""
    hash: str = ""


def build_demo_chain(n=5):
    rows, prev = [], trackrecord.GENESIS_HASH
    for i in range(n):
        row = _Row(
            f"demo:{i}",
            datetime(2026, 1, 1 + i),
            "fake-v1",
            "forecast_v1",
            "abc",
            0.5 + i * 0.01,
            0.5 + i * 0.01,
            "identity",
        )
        fields = trackrecord.hashable_fields(row)
        row.prev_hash = prev
        row.hash = trackrecord.compute_hash(prev, fields)
        rows.append(row)
        prev = row.hash
    return rows


chain = build_demo_chain()
print("untampered chain verifies:", trackrecord.verify_chain(chain) is None)

In [ ]:
chain[2].raw_prob = 0.99  # tamper with history after the fact
result = trackrecord.verify_chain(chain)
print("after editing forecast #2 without recomputing hashes:")
print("  break detected at index:", result.index if result else None)

## Scores against baselines

Once you have real resolved questions, compare this pipeline's Brier score against: always-50%, the base rate of your question set, and the market's own last price. Beating a liquid market consistently is hard — matching it with a cheap pipeline is already a strong result (guide §07).